# 22 - Conversao das notas fiscais pessoais para Parquets mensais

Objetivo: converter o CSV exportado manualmente com as notas fiscais pessoais do autor (`data/raw/notas-pessoais/notas-lote-1.csv`) em Parquets mensais no mesmo formato da base interim usada pelos notebooks 30 e 41.

- Origem: CSV criado pelo autor a partir das proprias NFC-e, com as mesmas 31 colunas dos CSVs do Portal da Transparencia (separador `;`, codificacao latin-1, virgula decimal).
- A base nao contem NCM, emitente, municipio nem UF; essas colunas ficam vazias.
- Todas as colunas sao lidas como texto, preservando zeros a esquerda e a virgula decimal, como nos Parquets interim originais.
- O mes de cada item e definido pela `DATA EMISSAO`. Nenhum mes e descartado, mesmo os que tem poucos itens.
- Saida: `data/interim/notas-pessoais/YYYY-MM.parquet`. A celula de gravacao falha se a pasta ja existir.
- Este e um experimento da branch `experimento-notas-pessoais`; o CSV bruto nao e versionado.

In [1]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

# Configuracoes da analise e caminhos relativos ao repositorio.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "notas-pessoais" / "notas-lote-1.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "notas-pessoais"
CSV_SEPARATOR = ";"
CSV_ENCODING = "latin-1"
DATE_FORMAT = "%d/%m/%Y %H:%M:%S"

In [2]:
def find_col(columns, *patterns):
    """Return the first column whose normalized name contains every pattern."""
    normalized_columns = {column: re.sub(r"\s+", " ", str(column)).upper() for column in columns}
    for column, normalized in normalized_columns.items():
        if all(pattern.upper() in normalized for pattern in patterns):
            return column
    raise KeyError(f"Nenhuma coluna encontrada para os padroes: {patterns}")


def load_source_csv(path):
    """Read the exported CSV as text, keeping empty fields as missing values."""
    frame = pd.read_csv(path, sep=CSV_SEPARATOR, encoding=CSV_ENCODING, dtype=str, keep_default_na=False)
    frame = frame.replace("", pd.NA)
    frame["arquivo_origem"] = path.name
    return frame

## 1. Carregar e validar o CSV

O CSV deve ter descricao e data de emissao em todas as linhas, e cada item (chave de acesso + numero do produto) deve aparecer uma unica vez.

In [3]:
items = load_source_csv(SOURCE_CSV_PATH)
desc_col = find_col(items.columns, "PRODUTO/SERV")
date_col = find_col(items.columns, "DATA", "EMISS")
key_col = find_col(items.columns, "CHAVE")
item_number_col = find_col(items.columns, "MERO PRODUTO")

emission_dates = pd.to_datetime(items[date_col], format=DATE_FORMAT)
assert items[desc_col].notna().all()
assert emission_dates.notna().all()
assert not items.duplicated([key_col, item_number_col]).any()

items["_mes"] = emission_dates.dt.strftime("%Y-%m")
non_empty = items.drop(columns=["_mes"]).notna().sum()

display(pd.DataFrame({
    "metrica": ["itens", "notas", "descricoes unicas", "primeira emissao", "ultima emissao"],
    "valor": [len(items), items[key_col].nunique(), items[desc_col].nunique(),
              emission_dates.min(), emission_dates.max()],
}))
display(non_empty[non_empty > 0].rename("valores_preenchidos").to_frame())
display(items["_mes"].value_counts().sort_index().rename("itens").to_frame())

,metrica,valor
0,itens,1931
1,notas,222
2,descricoes unicas,996
3,primeira emissao,2025-11-02 11:54:30
4,ultima emissao,2026-11-01 11:59:45


,valores_preenchidos
CHAVE DE ACESSO,1931
MODELO,1931
SÉRIE,1931
NÚMERO,1931
DATA EMISSÃO,1931
NÚMERO PRODUTO,1931
DESCRIÇÃO DO PRODUTO/SERVIÇO,1931
QUANTIDADE,1931
UNIDADE,1931
VALOR UNITÁRIO,1931


,itens
_mes,
2025-11,130
2025-12,188
2026-01,166
2026-02,137
2026-03,138
2026-04,186
2026-05,183
2026-06,206
2026-07,106


## 2. Gravar os Parquets mensais

**Esta celula grava arquivos** em `data/interim/notas-pessoais/` e falha se a pasta ja existir, para nao sobrescrever resultados anteriores.

In [4]:
if OUTPUT_DIR.exists():
    raise FileExistsError(
        f"A pasta de saida ja existe: {OUTPUT_DIR}. "
        "Escolha um novo nome em OUTPUT_DIR para preservar os arquivos existentes."
    )

OUTPUT_DIR.mkdir(parents=True)
generation_summary = []
for month, month_items in items.groupby("_mes", sort=True):
    output_path = OUTPUT_DIR / f"{month}.parquet"
    month_items.drop(columns=["_mes"]).to_parquet(output_path, index=False)
    generation_summary.append({"arquivo": output_path.name, "itens": len(month_items)})

generation_summary = pd.DataFrame(generation_summary)
assert generation_summary["itens"].sum() == len(items)
display(generation_summary)

,arquivo,itens
0,2025-11.parquet,130
1,2025-12.parquet,188
2,2026-01.parquet,166
3,2026-02.parquet,137
4,2026-03.parquet,138
5,2026-04.parquet,186
6,2026-05.parquet,183
7,2026-06.parquet,206
8,2026-07.parquet,106
9,2026-08.parquet,175
